# AIO-Mesh Rev1: Kaggle benchmark and ablation runner

This notebook runs the revision benchmark and ablation workflows for NSFNET, GEANT2, and Vietnam.

Attach a Kaggle Dataset containing a project snapshot built from the current repository. The snapshot must include the benchmark seed override (`evaluation_seeds` in `config.py`) and `data/real_telecom_combined.csv` (or `data/processed/real_telecom_combined.csv`). The notebook trains its own checkpoints from that snapshot; it does not use the old v11 checkpoints for evaluation. If you attach prior Kaggle outputs under a folder named results_kaggle, the notebook records their inventory separately and never mixes them into the new run.

Run the smoke cells first. They train disposable 512-step checkpoints and evaluate both pipelines for 64 requests. Smoke outputs only demonstrate that the workflow runs; do not use their metrics in the paper.

The full run is disabled by default. After smoke passes, set `RUN_FULL = True` in the configuration cell and run the full-run cell. Full execution trains production checkpoints, then runs 3000 requests per seed. Expect this stage to take substantially longer.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path

TOPOLOGIES = ["nsfnet", "geant2", "vietnam"]
EXPECTED_NODES = {"nsfnet": 14, "geant2": 23, "vietnam": 10}
EVAL_SEEDS = [42, 100, 2024, 8888, 9999, 7, 17, 123, 2025, 31415]
ABLATION_SCENARIOS = ["uniform", "bursty", "heavy_tail"]
BENCHMARK_SCENARIO = "thesis"  # normal, Elephant, burst-surge, Chaos
BENCHMARK_LOAD = "stress"
ABLATION_ARRIVAL_RATE = 1.0
TTL_MIN = 100
TTL_MAX = 500
TRAIN_SCENARIO = "heavy_tail"
SMOKE_TRAIN_STEPS = 512
SMOKE_EVAL_STEPS = 64
FULL_EVAL_STEPS = 3000
FULL_TRAIN_STEPS = {"vietnam": 500_000, "nsfnet": 2_000_000, "geant2": 2_000_000}
TRAIN_SEED = 42
TRAIN_ENVS = 4
RUN_FULL = False  # Flip to True only after all smoke checks pass.
SMOKE_PASSED = False
FULL_PASSED = False

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_ROOT = Path("/kaggle/working/aio_mesh_rev1") / RUN_ID
PROJECT_DIR = RUN_ROOT / "project"
SMOKE_ROOT = RUN_ROOT / "smoke"
FULL_ROOT = RUN_ROOT / "full"
SMOKE_MODELS = SMOKE_ROOT / "models"
FULL_MODELS = FULL_ROOT / "models"
print({"run_id": RUN_ID, "topologies": TOPOLOGIES, "seeds": EVAL_SEEDS, "full_enabled": RUN_FULL})


In [ ]:
import importlib.metadata
import os
import shutil
import subprocess
import sys
import zipfile

KAGGLE_INPUT = Path("/kaggle/input")
RUN_ROOT.mkdir(parents=True, exist_ok=True)

# Kaggle images vary; install only project runtime packages and record the resolved versions later.
packages = ["gymnasium", "stable-baselines3[extra]", "sb3-contrib", "shimmy>=0.2.1", "numpy", "pandas", "matplotlib", "networkx", "scipy"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


def is_project_root(path):
    required = [
        path / "src/analytics/benchmark/benchmark_algorithm/run_all.py",
        path / "src/analytics/training/train_harp_ablation.py",
        path / "src/analytics/ablation/ablation_study.py",
        path / "src/orchestration/jo_vdpr/env.py",
    ]
    return path.is_dir() and all(item.exists() for item in required)


def find_project_root(path):
    if is_project_root(path):
        return path
    if not path.exists():
        return None
    for candidate in path.rglob("run_all.py"):
        if candidate.as_posix().endswith("/src/analytics/benchmark/benchmark_algorithm/run_all.py"):
            for parent in candidate.parents:
                if is_project_root(parent):
                    return parent
    return None


def zip_has_project(path):
    try:
        with zipfile.ZipFile(path) as archive:
            return any(name.endswith("src/analytics/benchmark/benchmark_algorithm/run_all.py") for name in archive.namelist())
    except (OSError, zipfile.BadZipFile):
        return False

source_root = None
source_zip = next((p for p in KAGGLE_INPUT.rglob("*.zip") if zip_has_project(p)), None)
if source_zip is not None:
    staging = RUN_ROOT / "source_staging"
    staging.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(source_zip) as archive:
        archive.extractall(staging)
    source_root = find_project_root(staging)
    print("Using source archive:", source_zip)
else:
    for input_path in KAGGLE_INPUT.iterdir():
        source_root = find_project_root(input_path)
        if source_root is not None:
            break
if source_root is None:
    raise FileNotFoundError("Attach a source ZIP/folder containing the current AIO-Mesh benchmark, trainer, ablation runner, and JOVDPR environment.")

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
shutil.copytree(source_root, PROJECT_DIR)
config_source = (PROJECT_DIR / "src/analytics/benchmark/benchmark_algorithm/config.py").read_text(encoding="utf-8")
if "evaluation_seeds" not in config_source:
    raise RuntimeError("The attached package is stale: rebuild the Kaggle source dataset from the current workspace so --seeds is supported.")

DATA_PATH = next((p for p in [PROJECT_DIR / "data/real_telecom_combined.csv", PROJECT_DIR / "data/processed/real_telecom_combined.csv"] if p.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Expected real_telecom_combined.csv under data/ or data/processed/.")

RUN_ENV = os.environ.copy()
RUN_ENV["PYTHONPATH"] = str(PROJECT_DIR) + os.pathsep + RUN_ENV.get("PYTHONPATH", "")
for name in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    RUN_ENV[name] = "1"
RUN_ENV["MPLCONFIGDIR"] = "/tmp/matplotlib"

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print("Project:", PROJECT_DIR)
print("Data:", DATA_PATH)


In [ ]:
import hashlib
import json
import platform
from importlib.metadata import PackageNotFoundError


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def sha256_source_tree(root):
    digest = hashlib.sha256()
    for path in sorted((root / "src").rglob("*.py")):
        digest.update(path.relative_to(root).as_posix().encode())
        digest.update(path.read_bytes())
    return digest.hexdigest()


def run_cli(name, command, log_root):
    log_root.mkdir(parents=True, exist_ok=True)
    completed = subprocess.run(command, cwd=PROJECT_DIR, env=RUN_ENV, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    log_path = log_root / f"{name}.log"
    log_path.write_text(completed.stdout, encoding="utf-8")
    if completed.returncode:
        print("FAILED:", " ".join(command))
        print(completed.stdout[-12000:])
        raise RuntimeError(f"{name} failed with exit code {completed.returncode}; see {log_path}")
    print("PASS:", name, "log:", log_path)
    return completed.stdout

from src.orchestration.jo_vdpr.topology import TopologyManager

for topology in TOPOLOGIES:
    node_count = TopologyManager(topology).num_nodes
    assert node_count == EXPECTED_NODES[topology], f"{topology}: expected {EXPECTED_NODES[topology]} nodes, found {node_count}"
assert DATA_PATH.is_file() and DATA_PATH.stat().st_size > 0
assert len(EVAL_SEEDS) == len(set(EVAL_SEEDS)) and len(EVAL_SEEDS) >= 10

legacy_checkpoints = []
for path in sorted((PROJECT_DIR / "results/models/v11").glob("**/*.zip")) if (PROJECT_DIR / "results/models/v11").exists() else []:
    legacy_checkpoints.append({"path": str(path.relative_to(PROJECT_DIR)), "sha256": sha256_file(path)})

prior_candidates = [Path("/kaggle/working/results_kaggle"), Path("/kaggle/results_kaggle")]
prior_candidates.extend(KAGGLE_INPUT.rglob("results_kaggle"))
prior_results = next((p for p in prior_candidates if p.is_dir()), None)
prior_inventory = []
if prior_results is not None:
    for path in sorted(prior_results.rglob("*")):
        if path.is_file() and path.suffix.lower() in {".csv", ".json", ".pdf", ".png", ".zip"}:
            prior_inventory.append({"path": str(path.relative_to(prior_results)), "bytes": path.stat().st_size, "sha256": sha256_file(path)})
    print(f"Found prior Kaggle results: {prior_results} ({len(prior_inventory)} inventoried files; kept separate)")
else:
    print("No prior results_kaggle folder attached; this does not block the new run.")

PRECHECK = {
    "run_id": RUN_ID,
    "topologies": {topology: TopologyManager(topology).num_nodes for topology in TOPOLOGIES},
    "data_path": str(DATA_PATH.relative_to(PROJECT_DIR)),
    "data_sha256": sha256_file(DATA_PATH),
    "source_tree_sha256": sha256_source_tree(PROJECT_DIR),
    "legacy_v11_checkpoints_not_used": legacy_checkpoints,
    "prior_results_kaggle_path": str(prior_results) if prior_results else None,
    "prior_results_kaggle_inventory": prior_inventory,
    "python": sys.version,
    "platform": platform.platform(),
}
print(json.dumps(PRECHECK, indent=2))


## Smoke stage

This stage creates a fresh smoke model root and trains all five ablation checkpoints for each topology. It then runs one benchmark scenario and the strict ablation evaluator for 64 requests. The main benchmark and ablation runners share the newly trained HARP full checkpoint and normalizer, so this catches action/observation/checkpoint interface mismatches before any production-length training.


In [ ]:
import pandas as pd

SMOKE_LOGS = SMOKE_ROOT / "logs"
SMOKE_OUTPUT = SMOKE_ROOT / "results"
SMOKE_MODEL_ROOT = SMOKE_MODELS
SMOKE_TRAIN_ROOT = SMOKE_MODEL_ROOT / "ablation"
SMOKE_TRAIN_ROOT.mkdir(parents=True, exist_ok=True)

for topology in TOPOLOGIES:
    run_cli(
        f"train_{topology}",
        [sys.executable, "-m", "src.analytics.training.train_harp_ablation",
         "--variant", "all", "--topology", topology, "--scenario", TRAIN_SCENARIO,
         "--total-steps", str(SMOKE_TRAIN_STEPS), "--n-envs", "1", "--seed", str(TRAIN_SEED),
         "--data-path", str(DATA_PATH), "--output-root", str(SMOKE_TRAIN_ROOT)],
        SMOKE_LOGS,
    )
    for variant in ["harp_full", "harp_wo_gat", "harp_wo_hard_masking", "harp_soft_msd_penalty", "harp_wo_adaptive_penalty"]:
        model_path = SMOKE_TRAIN_ROOT / variant / f"dgrl_v11_{variant}_{topology}.zip"
        norm_path = SMOKE_TRAIN_ROOT / variant / f"vec_normalize_v11_{variant}_{topology}.pkl"
        assert model_path.is_file() and norm_path.is_file(), f"Missing smoke checkpoint/normalizer: {model_path}, {norm_path}"

    benchmark_dir = SMOKE_OUTPUT / "benchmark" / topology
    run_cli(
        f"benchmark_smoke_{topology}",
        [sys.executable, "-m", "src.analytics.benchmark.benchmark_algorithm.run_all",
         "--topology", topology, "--scenario", "uniform", "--steps", str(SMOKE_EVAL_STEPS),
         "--single-seed", "--skip-exhaustive", "--data-path", str(DATA_PATH),
         "--model-root", str(SMOKE_MODEL_ROOT), "--output-dir", str(benchmark_dir)],
        SMOKE_LOGS,
    )
    benchmark_files = list(benchmark_dir.rglob("all_seed_summary.csv"))
    assert benchmark_files, f"Benchmark produced no all_seed_summary.csv for {topology}"
    benchmark_rows = pd.concat([pd.read_csv(path) for path in benchmark_files], ignore_index=True)
    assert set(benchmark_rows["algorithm"]) >= {"harp", "saf_h"}
    assert set(benchmark_rows["topology"]) == {topology} and set(benchmark_rows["seed"]) == {42}

    ablation_dir = SMOKE_OUTPUT / "ablation" / topology
    run_cli(
        f"ablation_smoke_{topology}",
        [sys.executable, "-m", "src.analytics.ablation.ablation_study",
         "--topology", topology, "--scenario", TRAIN_SCENARIO, "--steps", str(SMOKE_EVAL_STEPS),
         "--seed", "42", "--arrival-rate", str(ABLATION_ARRIVAL_RATE), "--ttl-min", str(TTL_MIN), "--ttl-max", str(TTL_MAX),
         "--data-path", str(DATA_PATH), "--model-root", str(SMOKE_MODEL_ROOT),
         "--output-dir", str(ablation_dir)],
        SMOKE_LOGS,
    )
    summary_path = ablation_dir / "harp_ablation_summary.csv"
    ablation_rows = pd.read_csv(summary_path)
    assert len(ablation_rows) == 6, f"Expected 6 strict ablation rows for {topology}, got {len(ablation_rows)}"
    assert set(ablation_rows["topology"]) == {topology} and set(ablation_rows["seed"]) == {42}
    numeric = ablation_rows.select_dtypes(include="number")
    assert not numeric.isna().any().any(), f"Smoke ablation produced NaN metrics for {topology}"

SMOKE_PASSED = True
print("SMOKE PASSED for benchmark and strict ablation on all 3 topologies.")


In [ ]:
import importlib.metadata
import json
import shutil


def runtime_versions():
    versions = {}
    for package in ["gymnasium", "stable-baselines3", "sb3-contrib", "shimmy", "numpy", "pandas", "torch", "scipy"]:
        try:
            versions[package] = importlib.metadata.version(package)
        except PackageNotFoundError:
            versions[package] = None
    return versions


def checkpoint_hashes(model_root):
    hashes = {}
    if model_root.exists():
        for path in sorted(model_root.rglob("*.zip")):
            hashes[str(path.relative_to(model_root))] = sha256_file(path)
        for path in sorted(model_root.rglob("*.pkl")):
            hashes[str(path.relative_to(model_root))] = sha256_file(path)
    return hashes

SMOKE_MANIFEST = {
    **PRECHECK,
    "stage": "smoke",
    "smoke_passed": SMOKE_PASSED,
    "training_steps_per_variant": SMOKE_TRAIN_STEPS,
    "training_seed": TRAIN_SEED,
    "evaluation_steps": SMOKE_EVAL_STEPS,
    "evaluation_seeds": [42],
    "dependencies": runtime_versions(),
    "checkpoints": checkpoint_hashes(SMOKE_MODEL_ROOT),
}
(SMOKE_ROOT / "run_manifest.json").write_text(json.dumps(SMOKE_MANIFEST, indent=2), encoding="utf-8")
smoke_archive = shutil.make_archive(str(RUN_ROOT / f"aio_mesh_smoke_{RUN_ID}"), "zip", root_dir=SMOKE_ROOT)
print("Smoke archive:", smoke_archive)


## Full revision run (opt-in)

Do not run this cell until the smoke stage passes. It trains the full ablation variants once per topology, then reuses those current-source checkpoints for the main benchmark and strict ablation evaluations. The training seed is fixed; the ten evaluation seeds are matched across methods, scenarios, and topologies.

The benchmark `thesis` preset expands to normal load, Elephant heavy-tail, burst surge, and the separate Chaos generator. The ablation matrix retains uniform, bursty, and heavy-tail workloads. No smoke result is merged into full outputs.


In [ ]:
if not RUN_FULL:
    print("Full run is disabled. After smoke passes, set RUN_FULL=True in the config cell and run this cell.")
else:
    if not SMOKE_PASSED:
        raise RuntimeError("Full run blocked: execute and pass every smoke check first.")

    FULL_LOGS = FULL_ROOT / "logs"
    FULL_OUTPUT = FULL_ROOT / "results"
    FULL_TRAIN_ROOT = FULL_MODELS / "ablation"
    FULL_TRAIN_ROOT.mkdir(parents=True, exist_ok=True)

    for topology in TOPOLOGIES:
        run_cli(
            f"train_full_{topology}",
            [sys.executable, "-m", "src.analytics.training.train_harp_ablation",
             "--variant", "all", "--topology", topology, "--scenario", TRAIN_SCENARIO,
             "--total-steps", str(FULL_TRAIN_STEPS[topology]), "--n-envs", str(TRAIN_ENVS),
             "--seed", str(TRAIN_SEED), "--data-path", str(DATA_PATH),
             "--output-root", str(FULL_TRAIN_ROOT)],
            FULL_LOGS,
        )

    benchmark_summaries = []
    for topology in TOPOLOGIES:
        benchmark_dir = FULL_OUTPUT / "benchmark" / topology
        run_cli(
            f"benchmark_full_{topology}",
            [sys.executable, "-m", "src.analytics.benchmark.benchmark_algorithm.run_all",
             "--topology", topology, "--scenario", BENCHMARK_SCENARIO, "--load", BENCHMARK_LOAD, "--steps", str(FULL_EVAL_STEPS),
             "--seeds", *[str(seed) for seed in EVAL_SEEDS], "--skip-exhaustive",
             "--data-path", str(DATA_PATH), "--model-root", str(FULL_MODELS), "--output-dir", str(benchmark_dir)],
            FULL_LOGS,
        )
        summary_files = list(benchmark_dir.rglob("all_seed_summary.csv"))
        if not summary_files:
            raise RuntimeError(f"No full benchmark summary produced for {topology}")
        benchmark_summaries.extend(pd.read_csv(path) for path in summary_files)

    ablation_summaries = []
    for topology in TOPOLOGIES:
        for scenario in ABLATION_SCENARIOS:
            for seed in EVAL_SEEDS:
                output_dir = FULL_OUTPUT / "ablation" / topology / scenario / f"seed_{seed}"
                run_cli(
                    f"ablation_full_{topology}_{scenario}_seed_{seed}",
                    [sys.executable, "-m", "src.analytics.ablation.ablation_study",
                     "--topology", topology, "--scenario", scenario, "--steps", str(FULL_EVAL_STEPS),
                     "--seed", str(seed), "--arrival-rate", str(ABLATION_ARRIVAL_RATE),
                     "--ttl-min", str(TTL_MIN), "--ttl-max", str(TTL_MAX),
                     "--data-path", str(DATA_PATH), "--model-root", str(FULL_MODELS),
                     "--output-dir", str(output_dir)],
                    FULL_LOGS,
                )
                summary_path = output_dir / "harp_ablation_summary.csv"
                if not summary_path.is_file():
                    raise RuntimeError(f"Missing ablation summary: {summary_path}")
                ablation_summaries.append(pd.read_csv(summary_path))

    benchmark_raw = pd.concat(benchmark_summaries, ignore_index=True)
    ablation_raw = pd.concat(ablation_summaries, ignore_index=True)
    benchmark_raw.to_csv(FULL_OUTPUT / "benchmark_per_seed.csv", index=False)
    ablation_raw.to_csv(FULL_OUTPUT / "ablation_per_seed.csv", index=False)

    benchmark_aggregate = benchmark_raw.groupby(["topology", "scenario", "algorithm"], as_index=False).agg(
        n_seeds=("seed", "nunique"), acceptance_mean=("acceptance_rate", "mean"),
        acceptance_std=("acceptance_rate", "std"), msd_violation_mean=("msd_violation_rate", "mean"),
        avg_latency_mean_ms=("avg_latency_ms", "mean"), avg_latency_std_ms=("avg_latency_ms", "std"),
    )
    benchmark_aggregate.to_csv(FULL_OUTPUT / "benchmark_aggregate.csv", index=False)
    ablation_aggregate = ablation_raw.groupby(["topology", "scenario", "key"], as_index=False).agg(
        n_seeds=("seed", "nunique"), safe_acceptance_mean=("safe_acceptance_rate", "mean"),
        safe_acceptance_std=("safe_acceptance_rate", "std"), admitted_msd_mean=("admitted_msd_violation_rate", "mean"),
        avg_latency_mean_ms=("avg_latency_ms", "mean"), avg_latency_std_ms=("avg_latency_ms", "std"),
    )
    ablation_aggregate.to_csv(FULL_OUTPUT / "ablation_aggregate.csv", index=False)

    paired_rows = []
    rng = __import__("numpy").random.default_rng(2026)
    for (topology, scenario), group in benchmark_raw.groupby(["topology", "scenario"]):
        paired = group.pivot(index="seed", columns="algorithm", values="acceptance_rate").dropna(subset=["harp", "saf_h"])
        diff = (paired["harp"] - paired["saf_h"]).to_numpy(dtype=float)
        if len(diff) < 2:
            raise RuntimeError(f"Need at least two matched seeds for paired test: {topology}/{scenario}")
        if (__import__("numpy").abs(diff) == 0).all():
            p_value = 1.0
            rank_biserial = 0.0
        else:
            from scipy.stats import rankdata, wilcoxon
            p_value = float(wilcoxon(diff, zero_method="wilcox", alternative="two-sided").pvalue)
            nonzero = diff[diff != 0]
            ranks = rankdata(__import__("numpy").abs(nonzero))
            rank_biserial = float((__import__("numpy").sign(nonzero) * ranks).sum() / ranks.sum())
        bootstrap_index = rng.integers(0, len(diff), size=(10000, len(diff)))
        bootstrap_mean = diff[bootstrap_index].mean(axis=1)
        ci_low, ci_high = __import__("numpy").quantile(bootstrap_mean, [0.025, 0.975])
        paired_rows.append({"topology": topology, "scenario": scenario, "n_seeds": len(diff),
                            "mean_paired_acceptance_difference": float(diff.mean()), "ci95_low": float(ci_low),
                            "ci95_high": float(ci_high), "p_value": p_value, "rank_biserial": rank_biserial})

    order = sorted(range(len(paired_rows)), key=lambda i: paired_rows[i]["p_value"])
    running = 0.0
    count = len(order)
    for rank, index in enumerate(order):
        running = max(running, (count - rank) * paired_rows[index]["p_value"])
        paired_rows[index]["p_holm"] = min(1.0, running)
    pd.DataFrame(paired_rows).to_csv(FULL_OUTPUT / "harp_vs_saf_h_paired_statistics.csv", index=False)

    expected_benchmark_rows = len(TOPOLOGIES) * 4 * len(EVAL_SEEDS) * 4
    if len(benchmark_raw) != expected_benchmark_rows:
        raise RuntimeError(f"Benchmark matrix incomplete: expected {expected_benchmark_rows} rows, got {len(benchmark_raw)}")
    expected_ablation_rows = len(TOPOLOGIES) * len(ABLATION_SCENARIOS) * len(EVAL_SEEDS) * 6
    if len(ablation_raw) != expected_ablation_rows:
        raise RuntimeError(f"Ablation matrix incomplete: expected {expected_ablation_rows} rows, got {len(ablation_raw)}")

    FULL_PASSED = True
    print("Full benchmark and ablation matrix completed.")


In [ ]:
if RUN_FULL and FULL_PASSED:
    import json
    import shutil

    manifest = {
        **PRECHECK,
        "stage": "full",
        "completed": True,
        "benchmark_scenario_preset": BENCHMARK_SCENARIO,
        "benchmark_scenarios": sorted(benchmark_raw["scenario"].unique().tolist()),
        "ablation_scenarios": ABLATION_SCENARIOS,
        "evaluation_seeds": EVAL_SEEDS,
        "evaluation_steps": FULL_EVAL_STEPS,
        "training_seed": TRAIN_SEED,
        "training_steps_by_topology": FULL_TRAIN_STEPS,
        "training_envs": TRAIN_ENVS,
        "dependencies": runtime_versions(),
        "smoke_checkpoints": checkpoint_hashes(SMOKE_MODELS),
        "full_checkpoints": checkpoint_hashes(FULL_MODELS),
        "benchmark_rows": len(benchmark_raw),
        "ablation_rows": len(ablation_raw),
    }
    (FULL_ROOT / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    full_archive = shutil.make_archive(str(RUN_ROOT / f"aio_mesh_full_{RUN_ID}"), "zip", root_dir=FULL_ROOT)
    print("Full results archive:", full_archive)
else:
    print("No full archive created. Full run is disabled or has not completed successfully.")
